# Doctor v3 Training & Benchmarks — Google Colab

Trains `doctor/v3_base` and `doctor/v3` (with nurse data), then runs `benchmark_nurse_v3` and `compare_all_versions`.

---

## ⚠️ One-Time Drive Setup (do this before running any cells)

The MIMIC-IV data files and existing trained artifacts must be uploaded to **Google Drive** once. The notebook mounts Drive and symlinks those folders into the cloned repo — so `paths.py` resolves everything transparently.

### Required Drive structure

```
MyDrive/proiect_licenta/
├── data/
│   ├── mimic-iv-ed/
│   │   ├── triage.csv
│   │   ├── edstays.csv
│   │   ├── vitalsign.csv        (~115 MB)
│   │   ├── medrecon.csv         (~360 MB)
│   │   └── files_created/
│   │       └── categorized_diagnosis.csv
│   └── mimic-iv/
│       └── hosp/
│           ├── patients.csv
│           └── services.csv
└── artifacts/
    ├── triage/v1/     ← copy from local artifacts/triage/v1/
    ├── doctor/v1/     ← copy from local artifacts/doctor/v1/
    └── doctor/v2/     ← copy from local artifacts/doctor/v2/
    # doctor/v3_base/ and doctor/v3/ are written automatically by this notebook
```

> **Tip:** Use the [Google Drive desktop app](https://www.google.com/drive/download/) to sync your local `data/` and `artifacts/` folders directly — much faster than uploading via the browser.

---

## Estimated runtimes (Colab T4 GPU or CPU)

| Step | Time |
|------|------|
| Setup + install | ~3 min |
| train_doctor_v3 (v3 base) | ~30–45 min |
| train_nurse_v3 (v3 nurse) | ~45–60 min |
| benchmark_nurse_v3 | ~2–5 min |
| compare_all_versions | < 1 min |

---
## Section 1 — Environment Setup
*Run these cells at the start of every Colab session.*

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Clone the repository
# The clone contains only code — data/ and artifacts/ are gitignored.
# Adjust the URL to your GitHub repo before running.

REPO_URL   = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'  # ← EDIT THIS
CLONE_PATH = '/content/proiect_licenta'

import os, shutil

if os.path.exists(CLONE_PATH):
    print(f'Directory {CLONE_PATH} already exists — pulling latest changes.')
    os.system(f'git -C {CLONE_PATH} pull')
else:
    os.system(f'git clone {REPO_URL} {CLONE_PATH}')

print('Done.')

In [ ]:
# Cell 3: Symlink data/ and artifacts/ from Drive into the cloned repo
# This lets paths.py resolve everything correctly without any code changes.
# Adjust DRIVE_PROJECT if your Drive folder is named differently.

DRIVE_PROJECT = '/content/drive/MyDrive/proiect_licenta'  # ← EDIT IF NEEDED

for folder in ('data', 'artifacts'):
    src  = f'{DRIVE_PROJECT}/{folder}'
    dest = f'{CLONE_PATH}/{folder}'
    if os.path.islink(dest):
        print(f'Symlink already exists: {dest} → {os.readlink(dest)}')
    elif os.path.exists(dest):
        print(f'WARNING: {dest} is a real directory, not a symlink. Skipping.')
    else:
        os.symlink(src, dest)
        print(f'Created symlink: {dest} → {src}')

print('Symlinks ready.')

In [ ]:
# Cell 4: Install the proiect_licenta package (editable) + runtime dependencies
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('-e', CLONE_PATH)                        # installs the local package
pip('xgboost>=2.0.0')                        # XGBoost (may already be present in Colab)
pip('thefuzz[speedup]>=0.22.0')              # fuzzy drug-name matching

print('All dependencies installed.')

---
## Section 2 — Verify Environment
*Confirm all data files and pre-trained artifacts are reachable before starting long training runs.*

In [ ]:
# Cell 5: Check all required files and artifacts exist
from proiect_licenta.paths import (
    TRIAGE_CSV, EDSTAYS_CSV, PATIENTS_CSV, DIAGNOSIS_CSV,
    SERVICES_CSV, VITALSIGN_CSV, MEDRECON_CSV,
    TRIAGE_V1_DIR, DOCTOR_V1_DIR, DOCTOR_V2_DIR,
    DOCTOR_V3_BASE_DIR, DOCTOR_V3_DIR,
)

checks = {
    'data/mimic-iv-ed/triage.csv':              TRIAGE_CSV,
    'data/mimic-iv-ed/edstays.csv':             EDSTAYS_CSV,
    'data/mimic-iv/hosp/patients.csv':          PATIENTS_CSV,
    'data/.../categorized_diagnosis.csv':       DIAGNOSIS_CSV,
    'data/mimic-iv/hosp/services.csv':          SERVICES_CSV,
    'data/mimic-iv-ed/vitalsign.csv (~115 MB)': VITALSIGN_CSV,
    'data/mimic-iv-ed/medrecon.csv (~360 MB)':  MEDRECON_CSV,
    'artifacts/triage/v1/ (required input)':    TRIAGE_V1_DIR,
    'artifacts/doctor/v1/ (for comparison)':    DOCTOR_V1_DIR,
    'artifacts/doctor/v2/ (for comparison)':    DOCTOR_V2_DIR,
}

all_ok = True
for name, path in checks.items():
    ok = path.exists()
    status = '✓' if ok else '✗ MISSING'
    print(f'{status}  {name}')
    if not ok:
        all_ok = False

print()
if not all_ok:
    raise RuntimeError(
        'One or more required files/artifacts are missing.\n'
        'Check the Drive setup instructions in Section 0 before continuing.'
    )
print('All checks passed ✓  Ready to train.')

---
## Section 3 — Train Doctor v3 Base

Trains the v3 base model: **13-class diagnosis space** (catch-all `"Symptoms, Signs, Ill-Defined"` excluded) on the full ~102 K admitted-patient dataset. No nurse data — same feature set as v1.

**Inputs:** `triage.csv`, `edstays.csv`, `patients.csv`, `categorized_diagnosis.csv`, `services.csv`, `artifacts/triage/v1/*`  
**Outputs:** `artifacts/doctor/v3_base/{diagnosis_model,department_model,metadata}` — saved directly to Drive via symlink.

⏱ *Expected runtime: 30–45 min*

In [ ]:
# Cell 6: Train Doctor v3 base
import runpy

runpy.run_path(
    f'{CLONE_PATH}/src/proiect_licenta/training/train_doctor_v3.py',
    run_name='__main__',
)

In [ ]:
# Cell 7: Confirm v3_base artifacts were saved to Drive
import json

meta_path = DOCTOR_V3_BASE_DIR / 'metadata.json'
assert meta_path.exists(), 'v3_base training did not produce metadata.json — check logs above.'

meta = json.loads(meta_path.read_text())
print('v3_base metadata saved to Drive:')
print(json.dumps(meta, indent=2))

---
## Section 4 — Train Nurse v3

Trains the v3-with-nurse model: same 13-class label space as v3 base, but adds **snapshot vitals**, **medication flags**, and **longitudinal vitals / cardiac rhythm** from `vitalsign.csv` (Phase B).  
`vitalsign.csv` is read in 1 M-row chunks to stay within Colab's RAM limits.

**Inputs:** everything from v3 base, plus `vitalsign.csv` and `medrecon.csv`  
**Outputs:** `artifacts/doctor/v3/{diagnosis_model,department_model,metadata}` — saved to Drive.

⏱ *Expected runtime: 45–60 min*

In [ ]:
# Cell 8: Train Nurse v3
runpy.run_path(
    f'{CLONE_PATH}/src/proiect_licenta/training/train_nurse_v3.py',
    run_name='__main__',
)

In [ ]:
# Cell 9: Confirm v3 (nurse) artifacts were saved to Drive
meta_path = DOCTOR_V3_DIR / 'metadata.json'
assert meta_path.exists(), 'v3 nurse training did not produce metadata.json — check logs above.'

meta = json.loads(meta_path.read_text())
print('v3 (nurse) metadata saved to Drive:')
print(json.dumps(meta, indent=2))

---
## Section 5 — Benchmark Nurse v3

Compares **v3 base vs v3 nurse** on the same held-out 20 % test split (seed = 42).  
Prints accuracy, top-3/5 accuracy, Cohen's κ, per-class breakdown, and top-20 feature importances (nurse-specific features highlighted).

⏱ *Expected runtime: 2–5 min*

In [ ]:
# Cell 10: Benchmark — v3 base vs v3 nurse
runpy.run_path(
    f'{CLONE_PATH}/benchmarks/benchmark_nurse_v3.py',
    run_name='__main__',
)

---
## Section 6 — Compare All Versions

Four-way accuracy comparison across **v1 / v2 / v3 base / v3 nurse**.  
Reads only `metadata.json` from each artifact directory — very fast.

> **Note:** v1 & v2 use 14 diagnosis classes (catch-all included); v3 base & v3 nurse use 13 (catch-all excluded). Accuracy numbers are not directly comparable across the two label spaces — the script flags this in its output.

⏱ *Expected runtime: < 1 min*

In [ ]:
# Cell 11: Compare all versions (v1, v2, v3_base, v3_nurse)
runpy.run_path(
    f'{CLONE_PATH}/benchmarks/compare_all_versions.py',
    run_name='__main__',
)